# 🎨 ComfyUI + SDXL (Unrestricted / Photorealism) + Realistic LoRAs + OpenPose Setup

This ready-to-run notebook sets up:
1. **GPU Acceleration**: Compatible with Colab T4 / V100 / A100.
2. **ComfyUI Core & ComfyUI-Manager**.
3. **ControlNet Preprocessors (OpenPose / DWPose / DensePose)**.
4. **SDXL Models**:
   - **Juggernaut XL V9** (State of the art photorealism)
   - **Pony Diffusion V6 XL** (Top unrestricted base for flexible prompts & realistic styling)
5. **ControlNet SDXL Models**:
   - **thibaud/OpenPoseXL2** for direct pose guidance.
6. **Realistic LoRAs**:
   - **SDXL Offset / Detail LoRA** + **Real Skin/Detail enhancer**.
7. **Cloudflare Tunnel**: Instant public URL without login or token limits.

In [ ]:
#@title 1. Verify GPU Allocation
!nvidia-smi

In [ ]:
#@title 2. Install ComfyUI, Manager & ControlNet Aux Nodes
import os

%cd /content
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q xformers
!apt-get -y install aria2

# Install ComfyUI-Manager
%cd /content/ComfyUI/custom_nodes
if not os.path.exists('ComfyUI-Manager'):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git

# Install ControlNet Preprocessors
if not os.path.exists('comfyui_controlnet_aux'):
    !git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git
    !pip install -q -r comfyui_controlnet_aux/requirements.txt

if not os.path.exists('ComfyUI-Advanced-ControlNet'):
    !git clone https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet.git

print("✅ Core ComfyUI and custom nodes installed successfully!")

In [ ]:
#@title 3. Download SDXL Models, VAE, OpenPose ControlNet & Realistic LoRAs
import os

CHECKPOINT_DIR = '/content/ComfyUI/models/checkpoints'
LORA_DIR = '/content/ComfyUI/models/loras'
VAE_DIR = '/content/ComfyUI/models/vae'
CONTROLNET_DIR = '/content/ComfyUI/models/controlnet'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LORA_DIR, exist_ok=True)
os.makedirs(VAE_DIR, exist_ok=True)
os.makedirs(CONTROLNET_DIR, exist_ok=True)

# 1. SDXL Checkpoints (Photorealistic & Unrestricted Models)
print("📥 Downloading Juggernaut XL V9 (Photorealism standard)... ")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors -d {CHECKPOINT_DIR} -o Juggernaut-XL_v9.safetensors

print("📥 Downloading Pony Diffusion V6 XL (Unrestricted Base)... ")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/AstraliteHeart/pony-diffusion-v6-xl/resolve/main/v6_sd_xl_base.safetensors -d {CHECKPOINT_DIR} -o PonyDiffusionV6_XL.safetensors

# 2. Official SDXL VAE
print("📥 Downloading SDXL VAE... ")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors -d {VAE_DIR} -o sdxl_vae.safetensors

# 3. OpenPose SDXL ControlNet
print("📥 Downloading SDXL OpenPose ControlNet... ")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/thibaud/controlnet-openpose-sdxl-1.0/resolve/main/OpenPoseXL2.safetensors -d {CONTROLNET_DIR} -o OpenPoseXL2.safetensors

# 4. High-detail & Realism LoRAs
print("📥 Downloading Realistic & Detail LoRAs... ")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_offset_example-lora_1.0.safetensors -d {LORA_DIR} -o sdxl_offset_lora.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/nerijs/pastel-mix-lora/resolve/main/add_detail.safetensors -d {LORA_DIR} -o add_detail_xl.safetensors

print("\n✅ All Checkpoints, ControlNet Pose, VAE, and LoRAs are ready in ComfyUI!")

In [ ]:
#@title 4. Launch ComfyUI with Cloudflare Tunnel
# Download Cloudflare Tunnel client
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket

def tunnel_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    
    print("\n🚀 Launching Cloudflare Tunnel to expose ComfyUI...")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    for line in p.stderr:
        if "trycloudflare.com" in line:
            for part in line.split():
                if "trycloudflare.com" in part:
                    print(f"\n🔗 COMFYUI PUBLIC URL: {part.strip()}\n")
                    break

threading.Thread(target=tunnel_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI
# Run ComfyUI with fp16 VAE to prevent black images
!python main.py --listen 127.0.0.1 --port 8188 --fp16-vae --preview-method auto